# LangChain Expression Language (LCEL)

LCEL simplifies and brings transparency to constructing chains.

* **Input and Output Types:** It establishes a permitted set of input and output types.
* **Standard Methods:** It includes a range of standard methods, such as:
  * `invoke`: Calls the runnable on a single input.
  * `stream`: Calls it on a single input and streams back a response.
  * `batch`: Calls the runnable on a list of inputs.
  * **Async methods:** Corresponding async methods exist for all synchronous methods.
* **Dynamic Parameter Modification:** Allows changing parameters during runtime.
* **Schemas:** All runnables feature an input schema and an output schema.
* **Out-of-the-box Features:** It provides async, batch, and streaming support immediately.
* **Fallbacks:** You can easily attach fallbacks to LLMs or entire chains, which is useful when language models are unpredictable.
* **Parallelism:** LCEL syntax facilitates running time-consuming LLM calls in parallel.
* **Built-in Logging:** It natively logs steps, inputs, and outputs, which is vital as chains and agents become more complex.


In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai.api_key = os.environ['OPENAI_API_KEY']


In [2]:
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.output_parser import StrOutputParser #takes the chat message and converts it into a string


In [3]:
#we will create a simple chain
#first lets define the prompt template
prompt = ChatPromptTemplate.from_template(
    "Tell me a joke about {topic}"
)

#initialize the model
model = ChatOpenAI()
outputparser = StrOutputParser() #converts the raw text generated by llm to a more readable form

#define the chain
chain = prompt | model | outputparser

#now invoke the chain i.e. run the chain
chain.invoke({"topic": "Humans"})


'Why did the human bring a ladder to the bar?\n\nBecause they heard the drinks were on the house!'

## More complex chain

In [4]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import DocArrayInMemorySearch


## `.as_retriever()` method

`similarity_search()` is vector-store-specific. `.as_retriever()` converts it into a generic, swappable interface that every LangChain chain is written to expect — so chains don't need custom code per vector store, and you can swap `DocArrayInMemorySearch` for `Pinecone` later without changing anything downstream.


In [5]:
#first convert the texts to embeddings
vectorstore = DocArrayInMemorySearch.from_texts(
    ["Mallikarjun studies at PDA College of Engineering", "bears like to eat honey"],
    embedding = OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever() #generic interface


In [6]:
retriever.get_relevant_documents("In which college does Mallikarjun study?")


[Document(page_content='Mallikarjun studies at PDA College of Engineering'),
 Document(page_content='bears like to eat honey')]

In [7]:
retriever.get_relevant_documents("what does bear like to eat?")


[Document(page_content='bears like to eat honey'),
 Document(page_content='Mallikarjun studies at PDA College of Engineering')]

Let's try it with a chain.

In [8]:
prompt = ChatPromptTemplate.from_template(
    """
    answer the question based on the context.
    context: {context},
    question: {question}
    """
)


## `RunnableMap`

### What it is
`RunnableMap` lets you run several "runnables" (or plain functions, via lambdas) together, and bundle up their outputs into a single dictionary. Each key you give it maps to something that knows how to produce that key's value from the same input.

### Why we need it
Our prompt template needs two separate variables filled in - `context` and `question` - but our chain only receives one thing as input (the user's question dict). Something needs to take that single input and produce both of those pieces before it can reach the prompt.

### How it works
```python
RunnableMap(
    {
        "context": lambda x: retriever.get_relevant_documents(x["question"]),
        "question": lambda x: x["question"]
    }
)
```
When this runs on an input like `{"question": "..."}`:
* the whole input dict gets passed in as `x` to *every* lambda inside the map
* the `"context"` lambda takes `x`, pulls out `x["question"]`, and uses it to fetch relevant docs from the retriever
* the `"question"` lambda just passes the question straight through unchanged
* both results get collected into a single output dict: `{"context": [...], "question": "..."}`

That output dict is exactly the shape the prompt template needs (`{context}` and `{question}` placeholders), so it can be piped straight into `prompt | model | outputparser` next.


In [9]:
from langchain.schema.runnable import RunnableMap


In [10]:
chain = RunnableMap(
    {
        "context": lambda x: retriever.get_relevant_documents(x["question"]),
        "question": lambda x: x["question"]
    }
) | prompt| model| outputparser


In [11]:
chain.invoke({"question": "In which college Mallikarjun studies?"})


'Mallikarjun studies at PDA College of Engineering.'

Here is how the chain is working:

1. It executes the `RunnableMap` first, since that's the first thing in the chain. The dict with the `question` key gets passed in as `x`, then the `context` lambda extracts the relevant documents using `x["question"]`, and the `question` lambda just passes the question through.
2. Then the prompt template is called with that output. Since we used `|`, we're explicitly telling LangChain to connect these components sequentially - the output of one becomes the input of the next.


In [12]:
inputs =  RunnableMap(
    {
        "context": lambda x: retriever.get_relevant_documents(x["question"]),
        "question": lambda x: x["question"]
    }
)

inputs.invoke({"question": "In which College is Mallikarjun studying?"})


{'context': [Document(page_content='Mallikarjun studies at PDA College of Engineering'),
  Document(page_content='bears like to eat honey')],
 'question': 'In which College is Mallikarjun studying?'}

## Function calling using bind method

In [13]:
functions = [
    {
      "name": "weather_search",
      "description": "Search for weather given an airport code",
      "parameters": {
        "type": "object",
        "properties": {
          "airport_code": {
            "type": "string",
            "description": "The airport code to get the weather for"
          },
        },
        "required": ["airport_code"]
      }
    }
  ]


In [14]:
prompt = ChatPromptTemplate.from_messages(
    [("human", "{input}")]
)

model = ChatOpenAI(temperature = 0).bind(functions = functions)


### Why `from_messages` instead of `from_template` here?

`ChatPromptTemplate.from_template("...")` (what we used everywhere else) is really just a shortcut - under the hood it builds a single human message for you from one plain string.

`ChatPromptTemplate.from_messages([...])` is the more general/explicit version - it lets you build a prompt out of a list of `(role, content)` pairs, so you can construct multi-message prompts (e.g. a system message + a human message + a prior AI message, for few-shot examples or conversation history).

Here we only need one human message, so both approaches would honestly work - `from_messages([("human", "{input}")])` is just the more verbose way of writing the same single-message prompt. It's used here to show the more general syntax, since it becomes necessary once you need more than just one message.


In [15]:
runnable = prompt | model

runnable.invoke({"input":"what is the weather in sf?"})


AIMessage(content='', additional_kwargs={'function_call': {'name': 'weather_search', 'arguments': '{"airport_code":"SFO"}'}})

## Fallbacks

### What they are
A fallback is a backup chain (or model) that LangChain will automatically try if your primary chain throws an error. You attach a list of fallback chains with `.with_fallbacks([...])`, and if the first chain fails, LangChain retries with the next one in the list, and so on.

### Why they're useful
LLMs can be unpredictable - sometimes a smaller/cheaper/faster model messes up the output format (like the `gpt-3.5-turbo-instruct` example below, which returns text that *looks* like JSON but isn't valid JSON, so `json.loads` throws a `TypeError`). Instead of the whole app crashing, a fallback lets you gracefully retry with a more reliable (but maybe slower/pricier) chain, so the user still gets a working response.


In [16]:
from langchain.llms import OpenAI
import json


In [17]:
simple_model = OpenAI(
    temperature = 0,
    model = "gpt-3.5-turbo-instruct" #basic model
)

simple_chain = simple_model | json.loads


In [18]:
challenge = "write three poems in a json blob, where each poem is a json blob of a title, author, and first line"


In [19]:
#if we call just the model on the challenge
simple_model.invoke(challenge)


'\n\n{\n    "title": "Autumn Leaves",\n    "author": "Emily Dickinson",\n    "first_line": "The leaves are falling, one by one"\n}\n\n{\n    "title": "The Ocean\'s Song",\n    "author": "Pablo Neruda",\n    "first_line": "I hear the ocean\'s song, a symphony of waves"\n}\n\n{\n    "title": "A Winter\'s Night",\n    "author": "Robert Frost",\n    "first_line": "The snow falls softly, covering the ground"\n}'

The output is structured but is not valid JSON format, so if we try to run the chain we should get an error.

In [20]:
#error is to be expected
simple_chain.invoke(challenge)

JSONDecodeError: Extra data: line 9 column 1 (char 125)

In [21]:
model = ChatOpenAI(temperature = 0)
chain = model | outputparser | json.loads


In [22]:
chain.invoke(challenge)


{'poem1': {'title': 'The Rose',
  'author': 'Emily Dickinson',
  'firstLine': 'A rose by any other name would smell as sweet'},
 'poem2': {'title': 'The Road Not Taken',
  'author': 'Robert Frost',
  'firstLine': 'Two roads diverged in a yellow wood'},
 'poem3': {'title': 'Hope is the Thing with Feathers',
  'author': 'Emily Dickinson',
  'firstLine': 'Hope is the thing with feathers that perches in the soul'}}

This is working correctly because we're using a smarter model, and we're also converting the `AIMessage` to a string before `json.loads`.

Now let's try falling back to this chain if the first chain throws an error.

In [23]:
final_chain = simple_chain.with_fallbacks([chain]) #we can add a list of chains - if one fails it will try the next, and so on
final_chain.invoke(challenge)


{'poem1': {'title': 'The Rose',
  'author': 'Emily Dickinson',
  'firstLine': 'A rose by any other name would smell as sweet'},
 'poem2': {'title': 'The Road Not Taken',
  'author': 'Robert Frost',
  'firstLine': 'Two roads diverged in a yellow wood'},
 'poem3': {'title': 'Hope is the Thing with Feathers',
  'author': 'Emily Dickinson',
  'firstLine': 'Hope is the thing with feathers that perches in the soul'}}

`simple_chain` executed first and got an error, then the second chain (`chain`) got executed instead.

## Interface

In [24]:
prompt = ChatPromptTemplate.from_template(
    "Tell me a short joke about {topic}"
)
model = ChatOpenAI()
outputparser = StrOutputParser()

chain = prompt | model | outputparser


In [25]:
chain.invoke({"topic": "bears"})


"Why do bears have hairy coats?\n\nBecause they don't like to shave!"

In [26]:
chain.batch([{"topic": "Humans"}, {"topic": "Education system"}])


['Why did the human go to the seafood restaurant? To test his "mussel" strength!',
 'Why did the math book look sad?\nBecause it had too many problems.']

In [27]:
for t in chain.stream({"topic": "bees"}):
    print(t)



Why
 did
 the
 bee
 get
 married
?
 Because
 he
 found
 his
 honey
!



### Async method - why we need it

Every synchronous LCEL method (`invoke`, `batch`, `stream`) has an async counterpart (`ainvoke`, `abatch`, `astream`). `chain.ainvoke(...)` does the exact same thing as `chain.invoke(...)`, except it's non-blocking - it uses Python's `async`/`await` instead of blocking the whole program while it waits for the LLM API to respond.

This matters a lot for anything that needs to handle multiple requests at once - e.g. a web server or app backend handling several users' questions simultaneously. With a synchronous call, the whole program would sit idle waiting on one LLM call before it could even start the next one. With the async version, the program can kick off an LLM call and go do other work (like starting another user's request) while it waits for the response to come back.


In [28]:
response = await chain.ainvoke({"topic": "Tigers"})
response


"Why did the tiger wear a mask to the zoo? \nBecause he didn't want to be spotted!"